# MD-GRTN Phase 1 — PEMS08
**Changes:** `d_model=128` | `n_heads=4` | `num_layers=5` | `gru_layers=5` | `seq_len=12` | `batch_size=32` | 2-seq MDAF | GAT+GRU | Huber loss | AdamW | **BackNet pre-training** | **temporal position encoding** | noise augmentation | distance adjacency

**Architecture:** MDAF(BackNet pretrained) → MGRC(GAT+GRU ×5) → STFormer(spatial+temporal pos enc) ×5 → Prediction

**VRAM estimate:** ~11 GB / 15 GB T4  ✓


In [ ]:
import os, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

# ══════════════════════════════════════════════════
# SEED — fixes ALL randomness across sessions
# Run this first, every single session
# ══════════════════════════════════════════════════
SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
    os.environ['PYTHONHASHSEED'] = str(seed)
    print(f'Seed set: {seed} — results reproducible across sessions ✓')

set_seed()

print('PyTorch :', torch.__version__)
print('CUDA    :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU     :', torch.cuda.get_device_name(0))
    print('VRAM    :', round(torch.cuda.get_device_properties(0).total_memory/1e9,1), 'GB')

In [ ]:
class Config:
    data_path   = "/content/PEMS08.npz"
    num_nodes   = 170
    in_features = 3
    seq_len     = 16
    pred_len    = 12
    feature_idx = 0
    noise_std   = 0.0
    train_ratio = 0.7
    val_ratio   = 0.1

    d_model     = 96
    n_heads     = 4
    gat_layers  = 3
    temp_layers = 4
    dropout     = 0.15   # ↓ slightly less dropout — was too restrictive

    batch_size   = 64
    lr           = 5e-4  # ↑ slightly higher start
    epochs       = 300   # ↑ more epochs
    patience     = 40    # ↑ more patience
    weight_decay = 3e-4  # ↓ less L2 — model was underfitting slightly
    best_path    = "mdgrtn_best.pt"

cfg    = Config()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Config | d={cfg.d_model} | GAT={cfg.gat_layers} | Temp={cfg.temp_layers} | "
      f"seq={cfg.seq_len} | lr={cfg.lr} | {device}")


In [ ]:
def load_pems08(cfg):
    raw  = np.load(cfg.data_path)
    data = raw["data"].astype(np.float32)
    T, N, F = data.shape
    print(f"Shape: {data.shape}")
    mean_np = data.mean(axis=0)
    std_np  = data.std(axis=0) + 1e-8
    data_clean = (data - mean_np[None]) / std_np[None]
    feat_std_raw   = std_np[:, cfg.feature_idx].mean()
    norm_noise_std = cfg.noise_std / (feat_std_raw + 1e-8)
    if cfg.noise_std == 0:
        print("Noise disabled — clean inputs for both train and eval")
    else:
        print(f"Normalised noise σ≈{norm_noise_std:.4f}")
    A_dist = None
    for key in ("adjacency_matrix", "adj_mx", "adj"):
        if key in raw:
            A_dist = raw[key].astype(np.float32)
            print(f"Adjacency from \"{key}\"")
            break
    if A_dist is None:
        A_dist = np.eye(N, dtype=np.float32)
    nonzero = A_dist[A_dist > 0]
    sigma   = nonzero.std() if len(nonzero) > 0 else 1.0
    A_dist  = np.exp(-(A_dist**2) / (sigma**2 + 1e-8))
    np.fill_diagonal(A_dist, 0.0)
    A_dist  = A_dist / (A_dist.sum(1, keepdims=True) + 1e-8)
    return data_clean, mean_np, std_np, A_dist, norm_noise_std


class TrafficDataset(Dataset):
    def __init__(self, data_clean, seq_len, pred_len, feature_idx,
                 noise_std=0.0, split_start=0, split_end=None, training=False):
        self.data      = data_clean
        self.seq_len   = seq_len
        self.pred_len  = pred_len
        self.feat_idx  = feature_idx
        self.noise_std = noise_std
        self.training  = training
        T = len(data_clean)
        split_end = split_end if split_end is not None else T
        last_i    = split_end - seq_len - pred_len + 1
        self.indices = list(range(split_start, last_i))

    def __len__(self): return len(self.indices)

    def __getitem__(self, idx):
        i   = self.indices[idx]
        rec = self.data[i : i + self.seq_len].copy()
        y   = self.data[i + self.seq_len : i + self.seq_len + self.pred_len,
                        :, self.feat_idx].copy()
        if self.training and self.noise_std > 0:
            rec += np.random.randn(*rec.shape).astype(np.float32) * self.noise_std
        return torch.from_numpy(rec), torch.from_numpy(y)


def build_dataloaders(cfg):
    set_seed()
    data_clean, mean_np, std_np, A_dist, norm_noise = load_pems08(cfg)
    T  = len(data_clean)
    t1 = int(T * cfg.train_ratio)
    t2 = int(T * (cfg.train_ratio + cfg.val_ratio))
    kw    = dict(batch_size=cfg.batch_size, num_workers=0, pin_memory=False)
    ds_kw = dict(seq_len=cfg.seq_len, pred_len=cfg.pred_len,
                 feature_idx=cfg.feature_idx, noise_std=norm_noise,
                 data_clean=data_clean)
    dl_tr = DataLoader(TrafficDataset(**ds_kw, split_start=0,  split_end=t1, training=True),  shuffle=True,  **kw)
    dl_va = DataLoader(TrafficDataset(**ds_kw, split_start=t1, split_end=t2, training=False), shuffle=False, **kw)
    dl_te = DataLoader(TrafficDataset(**ds_kw, split_start=t2, split_end=T,  training=False), shuffle=False, **kw)
    print(f"Train={len(dl_tr.dataset)} | Val={len(dl_va.dataset)} | Test={len(dl_te.dataset)}")
    return dl_tr, dl_va, dl_te, mean_np, std_np, A_dist, norm_noise

print("Data utilities ready.")


In [ ]:
class InputProjection(nn.Module):
    """Simple clean projection: in_features → d_model.
    No noise, no dual branches — removes a source of training variance."""
    def __init__(self, in_dim, d_model, dropout=0.1):
        super().__init__()
        self.proj = nn.Linear(in_dim, d_model)
        self.norm = nn.LayerNorm(d_model)
        self.drop = nn.Dropout(dropout)
        self.ff   = nn.Sequential(
            nn.Linear(d_model, d_model * 2), nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * 2, d_model))
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):                  # (B, S, N, F)
        h = self.drop(self.proj(x))
        h = self.norm(h)
        return self.norm2(h + self.ff(h))  # (B, S, N, d)


class AdjacencyFusion(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.linear = nn.Linear(d_model, d_model)
        self.norm   = nn.LayerNorm(d_model)
    def forward(self, x, A):
        B, S, N, d = x.shape
        x_flat = x.reshape(B*S, N, d)
        neigh  = torch.einsum("nm,bmd->bnd", A, x_flat)
        out    = self.norm(x_flat + self.linear(neigh))
        return out.reshape(B, S, N, d)

print("InputProjection + AdjacencyFusion defined.")


In [ ]:
class GraphAttention(nn.Module):
    """Multi-head GAT using the fused adjacency as an attention mask."""
    def __init__(self, in_dim, out_dim, n_heads=4, dropout=0.1):
        super().__init__()
        assert out_dim % n_heads == 0
        self.n_heads = n_heads
        self.d_head  = out_dim // n_heads
        self.W       = nn.Linear(in_dim, out_dim, bias=False)
        # per-head attention vectors (size = d_head each)
        self.a_src   = nn.Parameter(torch.randn(1, n_heads, 1, self.d_head))
        self.a_dst   = nn.Parameter(torch.randn(1, n_heads, 1, self.d_head))
        self.drop    = nn.Dropout(dropout)
        self.out     = nn.Linear(out_dim, out_dim)
        nn.init.xavier_uniform_(self.W.weight)

    def forward(self, x, A):
        # x: (B, N, in_dim)   A: (N, N)
        B, N, _ = x.shape
        h = self.W(x).reshape(B, N, self.n_heads, self.d_head)  # (B,N,H,d)
        h = h.permute(0, 2, 1, 3)                                # (B,H,N,d)
        # e_src[b,h,i,1] = (h[b,h,i,:] * a_src).sum(-1)
        e_src = (h * self.a_src).sum(-1, keepdim=True)           # (B,H,N,1)
        # e_dst[b,h,1,j] = (h[b,h,j,:] * a_dst).sum(-1)
        e_dst = (h * self.a_dst).sum(-1, keepdim=True)           # (B,H,N,1)
        # broadcast: (B,H,N,1) + (B,H,1,N) → (B,H,N,N)
        e = F.leaky_relu(e_src + e_dst.permute(0, 1, 3, 2), negative_slope=0.2)
        # mask non-edges
        mask  = (A == 0).unsqueeze(0).unsqueeze(0)               # (1,1,N,N)
        e     = e.masked_fill(mask, -1e9)
        alpha = self.drop(torch.softmax(e, dim=-1))               # (B,H,N,N)
        out   = (alpha @ h).permute(0, 2, 1, 3).reshape(B, N, -1)  # (B,N,out_dim)
        return self.out(out)


class GATLayer(nn.Module):
    """Single GAT layer with residual + LayerNorm.
    Applied independently to each timestep: (B,N,d) → (B,N,d).
    No GRU — temporal dependencies handled by STFormer."""
    def __init__(self, d_model, n_heads=4, dropout=0.1):
        super().__init__()
        self.gat  = GraphAttention(d_model, d_model, n_heads, dropout)
        self.norm = nn.LayerNorm(d_model)
        self.ff   = nn.Sequential(
            nn.Linear(d_model, d_model * 2), nn.GELU(),
            nn.Linear(d_model * 2, d_model))
        self.norm2 = nn.LayerNorm(d_model)
        self.drop  = nn.Dropout(dropout)

    def forward(self, x, A):
        # x: (B, N, d)   A: (N, N)
        x = self.norm (x + self.drop(self.gat(x, A)))   # GAT + residual
        x = self.norm2(x + self.drop(self.ff(x)))        # FFN + residual
        return x


class MGRCModule(nn.Module):
    """Pure stacked GAT (no GRU, no attention beyond GAT).
    Each layer: fused-adj GAT → residual → LayerNorm → FFN → residual → LayerNorm.
    Applied across all timesteps independently — temporal context from STFormer."""
    def __init__(self, in_dim, hidden_dim, num_nodes, num_layers=4, n_heads=4, dropout=0.1):
        super().__init__()
        # Dynamic adjacency embedding (paper Eq.10)
        self.emb      = nn.Parameter(torch.randn(num_nodes, hidden_dim))
        self.adj_conv = nn.Conv2d(2, 1, kernel_size=1)
        # Input projection if dims differ
        self.input_proj = nn.Linear(in_dim, hidden_dim) if in_dim != hidden_dim else nn.Identity()
        # Stacked GAT layers — all same dimension
        self.layers = nn.ModuleList([
            GATLayer(hidden_dim, n_heads, dropout) for _ in range(num_layers)])

    def get_fused_adj(self, A_dist):
        A_dyna  = torch.softmax(self.emb @ self.emb.T, dim=-1)
        stacked = torch.stack([A_dist, A_dyna], dim=0).unsqueeze(0)  # (1,2,N,N)
        A_F     = F.relu(self.adj_conv(stacked).squeeze(0).squeeze(0))
        return A_F / (A_F.sum(-1, keepdim=True) + 1e-8)

    def forward(self, x, A_dist):
        # x: (B, S, N, d)
        B, S, N, d = x.shape
        A_F = self.get_fused_adj(A_dist)              # (N, N)
        x   = self.input_proj(x)                       # (B, S, N, hidden)
        # Apply each GAT layer across all timesteps independently
        x_flat = x.reshape(B*S, N, -1)                # (B*S, N, d)
        for layer in self.layers:
            x_flat = layer(x_flat, A_F)
        return x_flat.reshape(B, S, N, -1)             # (B, S, N, d)

print("MGRC defined — pure stacked GAT (no GRU).")

In [ ]:
class SinusoidalPE(nn.Module):
    """Fixed sinusoidal positional encoding (LLM-style, not learnable).
    Provides strong inductive bias for sequence order."""
    def __init__(self, d_model, max_len=512):
        super().__init__()
        pe  = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() *
                        (-torch.log(torch.tensor(10000.0)) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))   # (1, max_len, d)

    def forward(self, x):
        # x: (B*N, S, d)
        return x + self.pe[:, :x.size(1)]


class TemporalTransformerLayer(nn.Module):
    """Temporal transformer with LLM-style sinusoidal PE + learnable TPE."""
    def __init__(self, d_model, n_heads, dropout, seq_len, num_nodes):
        super().__init__()
        # LLM-style fixed sinusoidal PE
        self.sin_pe = SinusoidalPE(d_model)
        # Learnable hourly/daily/weekly weights (paper Eq.21)
        self.W_hour = nn.Parameter(torch.randn(num_nodes, 1) * 0.02)
        self.W_day  = nn.Parameter(torch.randn(num_nodes, 1) * 0.02)
        self.W_week = nn.Parameter(torch.randn(num_nodes, 1) * 0.02)
        t = torch.arange(seq_len).float()
        self.register_buffer("E_hour", (t % 12 + 1).unsqueeze(0))
        self.register_buffer("E_day",  (t % 24 + 1).unsqueeze(0))
        self.register_buffer("E_week", (t % 7  + 1).unsqueeze(0))
        self.tpe_proj = nn.Linear(1, d_model)
        # Transformer block
        self.attn  = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.ff    = nn.Sequential(nn.Linear(d_model, d_model*4), nn.GELU(),
                                   nn.Linear(d_model*4, d_model))
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.drop  = nn.Dropout(dropout)

    def forward(self, x):
        # x: (B, S, N, d)
        B, S, N, d = x.shape
        # Learnable temporal PE
        enc = (self.W_hour @ self.E_hour +
               self.W_day  @ self.E_day  +
               self.W_week @ self.E_week)         # (N, S)
        enc = self.tpe_proj(enc.T.unsqueeze(0).unsqueeze(-1))  # (1, S, N, d)
        x   = x + enc
        # Reshape for temporal attention
        f = x.permute(0,2,1,3).reshape(B*N, S, d)
        # LLM-style sinusoidal PE
        f = self.sin_pe(f)
        h, _ = self.attn(f, f, f)
        h = self.norm1(f + self.drop(h))
        h = self.norm2(h + self.drop(self.ff(h)))
        return h.reshape(B, N, S, d).permute(0,2,1,3)

print("Temporal transformer with sinusoidal PE defined.")


In [ ]:
class MDGRTN(nn.Module):
    """
    Clean architecture — no dual diffusion noise:
    x_rec → InputProjection → AdjFusion → GAT×4 → TemporalTransformer×4 → Output
    Smaller (d=64), stronger regularisation (dropout=0.2, wd=5e-4)
    """
    def __init__(self, cfg):
        super().__init__()
        N, F, d = cfg.num_nodes, cfg.in_features, cfg.d_model
        h, dr   = cfg.n_heads, cfg.dropout
        P, S    = cfg.pred_len, cfg.seq_len

        self.input_proj = InputProjection(F, d, dr)
        self.adj_fuse   = AdjacencyFusion(d)
        self.gat_stack  = MGRCModule(d, d, N, num_layers=cfg.gat_layers,
                                     n_heads=h, dropout=dr)
        self.temp_layers = nn.ModuleList([
            TemporalTransformerLayer(d, h, dr, S, N)
            for _ in range(cfg.temp_layers)])
        self.out_proj = nn.Linear(d * S, P)

    def forward(self, x_rec, A_dist):
        x = self.input_proj(x_rec)
        x = self.adj_fuse(x, A_dist)
        x = self.gat_stack(x, A_dist)
        for layer in self.temp_layers:
            x = layer(x)
        B, S, N, d = x.shape
        return self.out_proj(x.permute(0,2,1,3).reshape(B,N,S*d)).permute(0,2,1)


set_seed()
model = MDGRTN(cfg).to(device)
total = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model ready | Parameters: {total:,}")
print(f"  d={cfg.d_model} | GAT={cfg.gat_layers} | Temp={cfg.temp_layers} | dropout={cfg.dropout}")
with torch.no_grad():
    dummy = torch.randn(2, cfg.seq_len, cfg.num_nodes, cfg.in_features).to(device)
    adj   = torch.eye(cfg.num_nodes).to(device)
    out   = model(dummy, adj)
print(f"Output shape: {out.shape}  ✓")


In [ ]:
def masked_mae(pred, true, null_val=0.0):
    mask = (true != null_val).float()
    return (torch.abs(pred-true)*mask).sum() / (mask.sum()+1e-8)

def huber_loss(pred, true, delta=1.0, null_val=0.0):
    """Smooth L1 / Huber — less sensitive to outliers than MAE."""
    mask = (true != null_val).float()
    err  = torch.abs(pred - true)
    loss = torch.where(err < delta, 0.5 * err**2, delta * (err - 0.5 * delta))
    return (loss * mask).sum() / (mask.sum() + 1e-8)

def masked_rmse(pred, true, null_val=0.0):
    mask = (true != null_val).float()
    return torch.sqrt(((pred-true)**2*mask).sum() / (mask.sum()+1e-8))

def masked_mape(pred, true, low_thresh=10.0):
    """Mask near-zero flow to avoid div/0."""
    mask = (true.abs() > low_thresh).float()
    if mask.sum() < 1: return torch.tensor(0.0, device=pred.device)
    return (torch.abs((pred-true)/(true.abs()+1.0))*mask).sum() / mask.sum() * 100

print('Metrics defined.')

In [ ]:
# from google.colab import drive; drive.mount('/content/drive')
# cfg.data_path = '/content/drive/MyDrive/PEMS08.npz'

dl_train, dl_val, dl_test, mean_np, std_np, A_dist_np, norm_noise = build_dataloaders(cfg)
mean_flow = torch.from_numpy(mean_np[:, cfg.feature_idx]).to(device)
std_flow  = torch.from_numpy(std_np [:, cfg.feature_idx]).to(device)
A_dist    = torch.from_numpy(A_dist_np).to(device)
print(f"Train batches: {len(dl_train)} | Val: {len(dl_val)} | Test: {len(dl_test)}")


In [ ]:
# Mixed precision — halves VRAM for activations, 2x faster on T4
scaler = torch.amp.GradScaler('cuda')

def train_epoch(model, loader, optimizer, A_dist, device):
    model.train()
    total = 0.0
    for x_rec, y in loader:
        x_rec = x_rec.to(device)
        y     = y.to(device)
        with torch.amp.autocast('cuda'):
            pred = model(x_rec, A_dist)
            loss = huber_loss(pred, y)
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        scaler.step(optimizer)
        scaler.update()
        total += loss.item()
    return total / len(loader)


@torch.no_grad()
def eval_epoch(model, loader, A_dist, device, mean_flow, std_flow):
    model.eval()
    maes, rmses, mapes = [], [], []
    for x_rec, y in loader:
        x_rec  = x_rec.to(device)
        y      = y.to(device)
        with torch.amp.autocast('cuda'):
            pred = model(x_rec, A_dist)
        pred_d = pred.float() * std_flow[None,None,:] + mean_flow[None,None,:]
        y_d    = y.float()    * std_flow[None,None,:] + mean_flow[None,None,:]
        maes.append(masked_mae(pred_d, y_d).item())
        rmses.append(masked_rmse(pred_d, y_d).item())
        mapes.append(masked_mape(pred_d, y_d).item())
    return np.mean(maes), np.mean(rmses), np.mean(mapes)

print("Train/eval with mixed precision (AMP) defined.")


def train_epoch_onecycle(model, loader, optimizer, scheduler, A_dist, device):
    """OneCycleLR requires scheduler.step() after every batch."""
    model.train()
    total = 0.0
    for x_rec, y in loader:
        x_rec = x_rec.to(device)
        y     = y.to(device)
        with torch.amp.autocast("cuda"):
            pred = model(x_rec, A_dist)
            loss = huber_loss(pred, y)
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()   # ← per batch for OneCycleLR
        total += loss.item()
    return total / len(loader)


In [ ]:
# Simple checkpoint save — no gru_layers reference
def save_best(model, path):
    torch.save(model.state_dict(), path)
    print(f"Best model saved → {path}")

print("Checkpoint utility ready.")


In [ ]:
set_seed()

optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

# OneCycleLR: warms up then anneals — known to reach lower loss than cosine
# Uses all epochs budget efficiently
steps_per_epoch = len(dl_train)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr      = cfg.lr,
    epochs      = cfg.epochs,
    steps_per_epoch = steps_per_epoch,
    pct_start   = 0.1,        # 10% warmup
    div_factor  = 10,         # start lr = max_lr / 10
    final_div_factor = 1e4,   # end lr = max_lr / 10000
    anneal_strategy = "cos")

best_val_mae = float("inf")
patience_cnt = 0
history      = {"train_loss":[], "val_mae":[], "val_rmse":[], "val_mape":[]}

print(f"Training | d={cfg.d_model} | GAT={cfg.gat_layers} | Temp={cfg.temp_layers}")
print(f"OneCycleLR | max_lr={cfg.lr} | epochs={cfg.epochs} | patience={cfg.patience}")
print(f"Baseline → MAE=13.114 | RMSE=22.623 | MAPE=8.471%")
print("="*65)

for epoch in range(1, cfg.epochs + 1):
    # OneCycleLR steps per batch inside train_epoch
    train_loss = train_epoch_onecycle(model, dl_train, optimizer, scheduler, A_dist, device)
    val_mae, val_rmse, val_mape = eval_epoch(
        model, dl_val, A_dist, device, mean_flow, std_flow)

    history["train_loss"].append(train_loss)
    history["val_mae"].append(val_mae)
    history["val_rmse"].append(val_rmse)
    history["val_mape"].append(val_mape)

    tag = ""
    if val_mae < best_val_mae:
        best_val_mae = val_mae
        patience_cnt = 0
        save_best(model, cfg.best_path)
        tag = "  ← best ✓"
    else:
        patience_cnt += 1
        if patience_cnt >= cfg.patience:
            print(f"Early stopping at epoch {epoch}.")
            break

    lr_now = optimizer.param_groups[0]["lr"]
    if epoch % 5 == 0 or tag:
        print(f"Ep {epoch:03d} | Loss={train_loss:.4f} | "
              f"MAE={val_mae:.3f} RMSE={val_rmse:.3f} MAPE={val_mape:.2f}% "
              f"lr={lr_now:.1e}{tag}")

print(f"\nBest Val MAE: {best_val_mae:.3f}")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(history['train_loss'], color='steelblue', label='Train Loss')
axes[0].set_title('Training Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend()

axes[1].plot(history['val_mae'], color='tab:orange', label='Val MAE')
axes[1].axhline(13.114, color='red', ls='--', label='Baseline 13.114')
axes[1].set_title('Validation MAE'); axes[1].set_xlabel('Epoch'); axes[1].legend()

axes[2].plot(history['val_rmse'], color='tab:green', label='Val RMSE')
axes[2].axhline(22.623, color='red', ls='--', label='Baseline 22.623')
axes[2].set_title('Validation RMSE'); axes[2].set_xlabel('Epoch'); axes[2].legend()

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()

In [ ]:
# ══════════════════════════════════════════════════
# FINAL TEST — paper-style single averaged metric
# ══════════════════════════════════════════════════
model.load_state_dict(torch.load(cfg.best_path, map_location=device))

@torch.no_grad()
def paper_style_eval(model, loader, A_dist, device, mean_flow, std_flow):
    model.eval()
    all_pred, all_true = [], []
    for x_rec, y in loader:
        x_rec  = x_rec.to(device)
        y      = y.to(device)
        pred   = model(x_rec, A_dist)
        pred_d = pred * std_flow[None,None,:] + mean_flow[None,None,:]
        y_d    = y    * std_flow[None,None,:] + mean_flow[None,None,:]
        all_pred.append(pred_d.cpu())
        all_true.append(y_d.cpu())

    P = torch.cat(all_pred, dim=0)   # (total, 12, 170)
    Y = torch.cat(all_true, dim=0)

    mae  = torch.abs(P - Y).mean().item()
    rmse = ((P - Y)**2).mean().sqrt().item()
    mask = Y.abs() > 10.0
    mape = (torch.abs((P[mask]-Y[mask])/(Y[mask].abs()+1.0))).mean().item()*100

    print('\n' + '='*55)
    print('  TEST RESULTS  —  averaged over all 12 steps')
    print('='*55)
    print(f'  MAE  : {mae:.3f}    baseline: 13.114   Δ={mae-13.114:+.3f}')
    print(f'  RMSE : {rmse:.3f}    baseline: 22.623   Δ={rmse-22.623:+.3f}')
    print(f'  MAPE : {mape:.3f}%   baseline:  8.471%  Δ={mape-8.471:+.3f}%')
    print('='*55)
    return mae, rmse, mape

mae, rmse, mape = paper_style_eval(
    model, dl_test, A_dist, device, mean_flow, std_flow)

In [ ]:
@torch.no_grad()
def horizon_eval(model, loader, A_dist, device, mean_flow, std_flow):
    model.eval()
    buf = {h:{'mae':[],'rmse':[],'mape':[]} for h in [2,5,11]}
    for x_rec, y in loader:
        x_rec  = x_rec.to(device)
        y      = y.to(device)
        pred   = model(x_rec, A_dist)
        pred_d = pred * std_flow[None,None,:] + mean_flow[None,None,:]
        y_d    = y    * std_flow[None,None,:] + mean_flow[None,None,:]
        for h in buf:
            buf[h]['mae'].append(masked_mae(pred_d[:,h,:], y_d[:,h,:]).item())
            buf[h]['rmse'].append(masked_rmse(pred_d[:,h,:], y_d[:,h,:]).item())
            buf[h]['mape'].append(masked_mape(pred_d[:,h,:], y_d[:,h,:]).item())

    print(f'{"Horizon":>14} | {"MAE":>8} | {"RMSE":>8} | {"MAPE":>9}')
    print('-'*50)
    for h, lbl in zip([2,5,11], ['3-step (15min)','6-step (30min)','12-step (60min)']):
        m = {k:np.mean(v) for k,v in buf[h].items()}
        print(f'{lbl:>14} | {m["mae"]:>8.3f} | {m["rmse"]:>8.3f} | {m["mape"]:>8.2f}%')

horizon_eval(model, dl_test, A_dist, device, mean_flow, std_flow)